# 05-6. 날짜와 시간 — 풀이 검증

## Goal

한국 시간 KST(UTC+09:00)로 결과를 표시하고 UTC 비교값·원문을 보존한다. 시간대 변환으로 한국 날짜가 바뀌는 사례를 검증한다.

> 학습자용 TODO를 먼저 완성한 뒤 참고한다.


## Setup

fixture와 실행 환경을 확인한다.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("requirements.txt가 있는 저장소 루트에서 JupyterLab을 실행하세요.")


ROOT = find_project_root()
FIXTURE_DIR = ROOT / "fixtures" / "05-text-processing"

assert sys.version_info >= (3, 10)
assert FIXTURE_DIR.is_dir()

print("Python:", sys.version.split()[0])
print("실습 데이터:", FIXTURE_DIR)


from datetime import datetime, timedelta, timezone
import json

KST = timezone(timedelta(hours=9), name="KST")
lesson_time = datetime(2026, 8, 14, 10, 30, tzinfo=KST)
print("한국 시각:", lesson_time.isoformat())
print("시간대:", lesson_time.tzname(), lesson_time.utcoffset())
# 실제 현재 한국 시각: datetime.now(KST). 검증은 고정 시각을 사용한다.
fixture_path = FIXTURE_DIR / "timestamp-events.jsonl"
records = [json.loads(line) for line in fixture_path.read_text(encoding="utf-8").splitlines()]


## Steps

참고 구현을 실행한다.


In [ ]:
def parse_utc(value: str) -> datetime:
    if not isinstance(value, str):
        raise TypeError("timestamp는 문자열이어야 한다.")
    value = value.strip()
    if not value:
        raise ValueError("timestamp는 비어 있을 수 없다.")
    normalized = value[:-1] + "+00:00" if value.endswith("Z") else value
    try:
        parsed = datetime.fromisoformat(normalized)
    except ValueError as exc:
        raise ValueError(f"잘못된 timestamp: {value!r}") from exc
    if parsed.tzinfo is None or parsed.utcoffset() is None:
        raise ValueError("시간대 정보가 필요하다.")
    return parsed.astimezone(timezone.utc)


def format_kst(value: datetime) -> str:
    if value.tzinfo is None or value.utcoffset() is None:
        raise ValueError("시간대 정보가 필요하다.")
    return value.astimezone(KST).isoformat(timespec="seconds")


valid, errors = [], []
for record in records:
    try:
        valid.append({**record, "timestamp_utc": parse_utc(record["timestamp"])})
    except (KeyError, TypeError, ValueError) as exc:
        event = record.get("event") if isinstance(record, dict) else None
        errors.append({"event": event, "code": "INVALID_TIMESTAMP", "message": str(exc)})
for item in sorted(valid, key=lambda item: item["timestamp_utc"]):
    item["timestamp_kst"] = format_kst(item["timestamp_utc"])
    print(item["event"], "KST:", item["timestamp_kst"])


## Checks

경계값과 fixture 결과를 대조한다.


In [ ]:
assert len(valid) == 2 and len(errors) == 2
assert valid[0]["timestamp_utc"].isoformat() == "2026-08-14T01:30:00+00:00"
assert all(item["timestamp_utc"].tzinfo == timezone.utc for item in valid)
assert valid[0]["timestamp_kst"] == "2026-08-14T10:30:00+09:00"
late_utc = parse_utc("2026-08-14T18:30:00Z")
assert format_kst(late_utc) == "2026-08-15T03:30:00+09:00"
next_day_kst = datetime.fromisoformat(format_kst(late_utc))
assert next_day_kst == late_utc
assert next_day_kst.timestamp() == late_utc.timestamp()
assert next_day_kst.astimezone(timezone.utc) == late_utc
assert next_day_kst.date().isoformat() == "2026-08-15"
assert next_day_kst.utcoffset() == timedelta(hours=9)
assert lesson_time.tzname() == "KST"
try:
    format_kst(datetime(2026, 8, 14, 10, 30))
except ValueError:
    pass
else:
    raise AssertionError("표시 함수가 시간대 없는 값을 허용했다.")
assert [item["event"] for item in sorted(valid, key=lambda item: item["timestamp_utc"])] == ["login", "api"]
for invalid in ("", "   ", 123, "2026-08-14ZT01:30:00"):
    try:
        parse_utc(invalid)
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError("타입·빈 값·Z 위치 경계를 허용했다.")
print("검증 통과")


## Next Steps

한국 시간 출력은 `astimezone(KST)`로 변환하고 `+09:00`을 표시한다. UTC는 여러 출처의 로그를 비교하는 기준이며 원문도 보존한다. 한국 날짜별 집계는 KST로 변환한 다음 수행한다. 과거 지역 규칙이 필요하면 `ZoneInfo('Asia/Seoul')`를 사용한다. 교안: `05-text-processing/05-6-datetime.md`.
